In [6]:
!pip install -q chromadb sentence-transformers

import chromadb
from chromadb.utils import embedding_functions

client = chromadb.Client()

collection = client.get_or_create_collection(
    name="my_first_collection"
)

documents = [
    "Python is a popular programming language for AI.",
    "The weather today is sunny and warm.",
    "Machine learning models learn patterns from data.",
    "I enjoy playing football on weekends.",
    "Deep learning uses neural networks with many layers.",
    "Paris is the capital city of France."
]

ids = ["doc1", "doc2", "doc3", "doc4", "doc5", "doc6"]

collection.upsert(
    documents=documents,
    ids=ids
)

print("ChromaDB client created successfully!")
print("Documents added!")
print("Total documents:", collection.count())

results = collection.query(
    query_texts=["Tell me about AI and neural networks"],
    n_results=2
)

print("\nSemantic Search Results:")

for doc, distance in zip(
    results["documents"][0],
    results["distances"][0]
):
    print(f"Distance: {distance:.4f} | {doc}")

results = collection.query(
    query_texts=["What is the capital of France?"],
    n_results=2
)

print("\nFrance Query Results:")

for doc, distance in zip(
    results["documents"][0],
    results["distances"][0]
):
    print(f"Distance: {distance:.4f} | {doc}")

collection.upsert(
    documents=[
        "The new smartphone has an amazing camera.",
        "Stock prices rose sharply after the earnings report."
    ],
    metadatas=[
        {"category": "technology"},
        {"category": "finance"}
    ],
    ids=["doc7", "doc8"]
)

results = collection.query(
    query_texts=["Tell me about new gadgets"],
    n_results=2,
    where={"category": "technology"}
)

print("\nMetadata Filter Results:")

for doc, distance in zip(
    results["documents"][0],
    results["distances"][0]
):
    print(f"Distance: {distance:.4f} | {doc}")

hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

custom_collection = client.get_or_create_collection(
    name="custom_embedding_collection",
    embedding_function=hf_embedder
)

custom_collection.upsert(
    documents=[
        "I love dogs",
        "I like puppies",
        "The stock market crashed today"
    ],
    ids=["a1", "a2", "a3"]
)

results = custom_collection.query(
    query_texts=["puppy lovers"],
    n_results=2
)

print("\nHugging Face Embedding Results:")

for doc, distance in zip(
    results["documents"][0],
    results["distances"][0]
):
    print(f"Distance: {distance:.4f} | {doc}")

collection.update(
    ids=["doc4"],
    documents=[
        "I enjoy playing football and basketball on weekends."
    ]
)

print("\nUpdated doc4")

collection.delete(
    ids=["doc2"]
)

print("Deleted doc2")
print("Remaining documents:", collection.count())

persistent_client = chromadb.PersistentClient(
    path="/content/chroma_db"
)

persistent_collection = persistent_client.get_or_create_collection(
    name="persistent_demo"
)

persistent_collection.upsert(
    documents=[
        "This data will be saved to disk."
    ],
    ids=["p1"]
)

print("\nPersistent Storage:")
print("Saved to: /content/chroma_db")
print("Count:", persistent_collection.count())

print("\nChromaDB demonstration completed successfully!")

ChromaDB client created successfully!
Documents added!
Total documents: 8

Semantic Search Results:
Distance: 0.9317 | Deep learning uses neural networks with many layers.
Distance: 0.9516 | Python is a popular programming language for AI.

France Query Results:
Distance: 0.3515 | Paris is the capital city of France.
Distance: 1.7959 | Python is a popular programming language for AI.

Metadata Filter Results:
Distance: 1.1460 | The new smartphone has an amazing camera.

Hugging Face Embedding Results:
Distance: 0.2513 | I like puppies
Distance: 0.3943 | I love dogs

Updated doc4
Deleted doc2
Remaining documents: 7

Persistent Storage:
Saved to: /content/chroma_db
Count: 1

ChromaDB demonstration completed successfully!
